# **Training GLiNER model with MD-related simulation's description** ⚙️

[GLiNER2](https://github.com/fastino-ai/GLiNER2) unifies Named Entity Recognition, Text Classification, Structured Data Extraction, and Relation Extraction into a single 205M parameter model. It provides efficient CPU-based inference without requiring complex pipelines or external API dependencies.

- See Documentation: https://urchade.github.io/GLiNER/training.html
- See Preprint: https://arxiv.org/abs/2311.08526


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

from dotenv import load_dotenv
from gliner2 import GLiNER2
from gliner2.training.data import InputExample
from gliner2.training.trainer import GLiNER2Trainer, TrainingConfig

from mdner_llm.core.logger import create_logger
from mdner_llm.models.entities_with_positions import ListOfEntitiesPositions
from mdner_llm.utils.evaluate_gliner2 import compare_entities
from mdner_llm.utils.select_annotation_files import select_annotation_files
from mdner_llm.utils.visualize_annotations import (
    convert_ner_response_to_entities,
    visualize_llm_annotation,
)

/data/touami/mdner_llm/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load models


In [2]:
# Load model with 205M parameters (base version)
base_extractor = GLiNER2.from_pretrained("fastino/gliner2-base-v1")
print(base_extractor)

🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first
GLiNER2(
  (encoder): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128011, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-11): 12 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
    

In [3]:
# Load model with 340M parameters (large version)
# large_extractor = GLiNER2.from_pretrained("fastino/gliner2-large-v1")
# print(large_extractor)

## Define entities classes


In [5]:
# Class of entities
entities_class_with_description = {
    "MOL": "Molecule or chemical compound involved in the simulation",
    "SOFTNAME": "Molecular dynamics software used for the simulation",
    "SOFTVERS": "Version of the molecular dynamics software",
    "TEMP": "Simulation temperature, typically expressed in Kelvin or Celcius",
    "FFM": "Force field model used to describe interatomic interactions",
    "STIME": "Total simulation time or duration",
}

## Prepare training data

In [ ]:
# Load annotations
selected_annotation_paths = select_annotation_files(
    annotations_dir=Path("../../annotations/v3"),
    nb_files=50,
    logger=create_logger()
)

2026-03-04 10:50:01.814 | INFO     | mdner_llm.utils.select_annotation_files:select_annotation_files:88 - Selecting text to annotate from ../../annotations/v3.
2026-03-04 10:50:01.817 | SUCCESS  | mdner_llm.utils.count_entities:list_json_files:55 - Found 360 JSON annotation files.


2026-03-04 10:50:01.829 | INFO     | mdner_llm.utils.select_annotation_files:select_annotation_files:130 - First annotation file path: ../../annotations/v3/zenodo_34415.json
2026-03-04 10:50:01.830 | SUCCESS  | mdner_llm.utils.select_annotation_files:select_annotation_files:131 - Selected 50 interesting annotations successfully!


In [7]:
# Create dataset for training

In [8]:
# Split into train/validation

## Configure training

In [9]:
model = GLiNER2.from_pretrained("fastino/gliner2-base-v1")
config = TrainingConfig(
    output_dir="../results/ner_model",
    experiment_name="gliner2_finetuned_50_descriptions",

    # Training steps
    num_epochs=10,
    max_steps=-1,

    # Batch size
    batch_size=32,
    eval_batch_size=64,
    gradient_accumulation_steps=1,

    # Learning rates
    encoder_lr=5e-6,  # Lower LR for fine-tuning
    task_lr=5e-4,

    # Early stopping
    early_stopping=False,
    early_stopping_patience=3,
    early_stopping_threshold=0.0,

    # Mixed precision
    fp16=True,

    # Checkpointing & Evaluation (saves when evaluating)
    eval_strategy="epoch",  # "epoch", "steps", or "no"
    metric_for_best="eval_loss",
    save_best=True,

    # Logging
    logging_steps=50,
)

🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first


## Train

In [10]:
trainer = GLiNER2Trainer(model, config)
trainer.train(train_data=train_data, val_data=val_data)

2026-03-04 10:50:03 - INFO - gliner2.training.trainer - LoRA is disabled


NameError: name 'train_data' is not defined

## Load best model

In [ ]:
model = GLiNER2.from_pretrained("./gliner2_finetuned_50_descriptions/best")
model